# LeetCode 2105 — Watering Plants II bằng Python

Link đề: https://leetcode.com/problems/watering-plants-ii/ · Độ khó: **Medium** · Chủ đề: Array, Two Pointers, Simulation

Skeleton LeetCode:

```python
class Solution:
    def minimumRefill(self, plants: List[int], capacityA: int, capacityB: int) -> int:
        ...
```

Lộ trình:
1. **Mô phỏng theo thời gian**: mỗi "giây", Alice và Bob mỗi người tưới một cây (nếu còn); xử lý va chạm ở cây giữa — sát nghĩa đen.
2. **Hai con trỏ** `i` (Alice, từ trái) và `j` (Bob, từ phải) tiến vào nhau — `O(n)` (**nộp bài**).
3. **Tách hai nửa độc lập + xử lý cây giữa**: dùng lại hàm đếm số lần nạp của bài 2079 cho từng nửa.
4. **Mô phỏng tổng quát hóa** thành hàm "một người tưới một dãy" trả về `(số lần nạp, nước còn lại)` — nền cho các biến thể.

## 1. Mục tiêu học

1. Đọc kỹ luật mô phỏng hai người: tưới **đồng thời**, nạp **ngay lập tức** khi thiếu, quy tắc ai tưới cây ở giữa.
2. Nhận ra hai người **không ảnh hưởng nhau** ngoại trừ cây giữa (khi `n` lẻ) ⇒ hai con trỏ đơn giản.
3. Cài đặt quy tắc phá hòa: so sánh **lượng nước hiện có** (không phải dung tích), hòa thì Alice tưới.
4. Tránh lỗi đếm cây giữa hai lần, và lỗi `<` vs `≤` khi quyết định nạp.
5. Kiểm chứng hai con trỏ bằng mô phỏng theo thời gian trên dữ liệu ngẫu nhiên.

## 2. Đề bài & ví dụ

`n` cây xếp hàng, cây `i` cần `plants[i]` đơn vị nước. Alice có bình dung tích `capacityA`, Bob có bình dung tích `capacityB`, ban đầu đều đầy.
- Alice tưới từ trái sang phải bắt đầu ở cây `0`; Bob tưới từ phải sang trái bắt đầu ở cây `n − 1`; **đồng thời**, mỗi cây mất cùng một khoảng thời gian.
- Mỗi người phải tưới nếu đủ nước; nếu không, **nạp đầy ngay lập tức** rồi tưới.
- Nếu cả hai tới **cùng một cây**: người có **nhiều nước hơn** (hiện tại) tưới; bằng nhau thì **Alice** tưới.

Trả về tổng số lần nạp nước.

**Ví dụ 1:** `[2,2,3,3]`, `5`, `5` → `1`. **Ví dụ 2:** `[2,2,3,3]`, `3`, `4` → `2`. **Ví dụ 3:** `[5]`, `10`, `8` → `0`.

**Ràng buộc:** `1 ≤ n ≤ 10⁵`, `1 ≤ plants[i] ≤ 10⁶`, `max(plants) ≤ capacityA, capacityB ≤ 10⁹`.

In [1]:
import random
import time
from typing import List

examples = [(([2, 2, 3, 3], 5, 5), 1), (([2, 2, 3, 3], 3, 4), 2), (([5], 10, 8), 0)]
for args, want in examples:
    print(args, "->", want)

([2, 2, 3, 3], 5, 5) -> 1
([2, 2, 3, 3], 3, 4) -> 2
([5], 10, 8) -> 0


## 3. Kiến thức nền

### 3.1 Một người tưới một dãy cây (bài 2079 rút gọn)

Đếm số lần nạp: thiếu nước (`water < need`) thì nạp đầy (`water = cap`), rồi tưới. Hàm trả thêm lượng nước còn lại để dùng cho cây giữa.

In [2]:
def water_sequence(seq, cap, water=None):
    """Tưới lần lượt các cây trong seq; trả về (số lần nạp, nước còn lại)."""
    water = cap if water is None else water
    refills = 0
    for need in seq:
        if water < need:
            refills += 1
            water = cap
        water -= need
    return refills, water

print(water_sequence([2, 2], 5), water_sequence([3, 3], 5), water_sequence([2, 2], 3))

(0, 1) (1, 2) (1, 1)


### 3.2 Hai con trỏ tiến vào nhau

`i` bắt đầu ở `0`, `j` ở `n − 1`; vòng lặp `while i < j` xử lý từng cặp; sau vòng, nếu `i == j` (n lẻ) còn **đúng một** cây giữa.

In [3]:
for n in (1, 2, 3, 4, 5):
    i, j, pairs = 0, n - 1, []
    while i < j:
        pairs.append((i, j)); i += 1; j -= 1
    print(f"n = {n}: các cặp {pairs}, cây giữa: {i if i == j else 'không có'}")

n = 1: các cặp [], cây giữa: 0
n = 2: các cặp [(0, 1)], cây giữa: không có
n = 3: các cặp [(0, 2)], cây giữa: 1
n = 4: các cặp [(0, 3), (1, 2)], cây giữa: không có
n = 5: các cặp [(0, 4), (1, 3)], cây giữa: 2


## 4. Ý tưởng nền tảng / trực giác

Vì tưới **đồng thời** và mỗi cây cùng thời gian, ở giây thứ `t` Alice tưới cây `t`, Bob tưới cây `n − 1 − t`. Hai người chỉ "gặp" nhau khi `t = n − 1 − t`, tức cây giữa của `n` lẻ. Với `n` chẵn, họ không bao giờ cùng cây.

Ví dụ 2 (`[2,2,3,3]`, `A = 3`, `B = 4`):

| giây | Alice: cây / nước trước / nạp? / nước sau | Bob: cây / nước trước / nạp? / nước sau |
|---|---|---|
| 0 | 0 / 3 / không / 1 | 3 / 4 / không / 1 |
| 1 | 1 / 1 / **có** / 1 | 2 / 1 / **có** / 1 |

Tổng `2` lần nạp ✓.

Ví dụ `[1, 2, 4, 2, 1]`, `A = 4`, `B = 5` (cây giữa là 2): sau hai giây Alice còn `4 − 1 − 2 = 1`, Bob còn `5 − 1 − 2 = 2`. Bob nhiều nước hơn ⇒ Bob tưới cây giữa (cần 4 > 2) ⇒ Bob nạp ⇒ tổng `1`.

In [4]:
def show_meeting(plants, A, B):
    n = len(plants); m = n // 2
    ra, wa = water_sequence(plants[:m], A)
    rb, wb = water_sequence(plants[n - m:][::-1], B)
    print(f"plants = {plants}: Alice tưới {plants[:m]} -> nạp {ra}, còn {wa}; Bob tưới {plants[n - m:][::-1]} -> nạp {rb}, còn {wb}")
    if n % 2:
        who = "Alice" if wa >= wb else "Bob"
        print(f"  cây giữa {m} cần {plants[m]}: {who} tưới (nước {max(wa, wb)}) -> {'nạp' if max(wa, wb) < plants[m] else 'không nạp'}")

show_meeting([1, 2, 4, 2, 1], 4, 5)
show_meeting([2, 2, 3, 3], 3, 4)

plants = [1, 2, 4, 2, 1]: Alice tưới [1, 2] -> nạp 0, còn 1; Bob tưới [1, 2] -> nạp 0, còn 2
  cây giữa 2 cần 4: Bob tưới (nước 2) -> nạp
plants = [2, 2, 3, 3]: Alice tưới [2, 2] -> nạp 1, còn 1; Bob tưới [3, 3] -> nạp 1, còn 1


## 5. Cách 1 — Brute Force / baseline: mô phỏng theo thời gian

**Ý tưởng:** sát nghĩa đen — mỗi bước thời gian, xác định cây mà Alice và Bob **định** tưới (cây chưa tưới kế tiếp ở mỗi phía). Nếu trùng nhau thì áp dụng quy tắc phá hòa. Lưu mảng `watered` để không tưới hai lần.

In [5]:
def refill_time_sim(plants, capacityA, capacityB):
    n = len(plants); watered = [False] * n
    wa, wb = capacityA, capacityB; ia, ib = 0, n - 1; refills = 0
    def do(water, cap, need):
        nonlocal refills
        if water < need:
            refills += 1; water = cap
        return water - need
    while ia < n and ib >= 0 and not all(watered):
        while ia < n and watered[ia]: ia += 1
        while ib >= 0 and watered[ib]: ib -= 1
        if ia >= n or ib < 0:
            break
        if ia == ib:                                    # cùng một cây
            if wa >= wb:
                wa = do(wa, capacityA, plants[ia])
            else:
                wb = do(wb, capacityB, plants[ib])
            watered[ia] = True
            break
        if ia > ib:
            break
        wa = do(wa, capacityA, plants[ia]); watered[ia] = True
        wb = do(wb, capacityB, plants[ib]); watered[ib] = True
    return refills

for args, want in examples:
    print(refill_time_sim(*args), "kỳ vọng", want)

1 kỳ vọng 1
2 kỳ vọng 2
0 kỳ vọng 0


**Độ phức tạp:** mỗi bước gọi `all(watered)` tốn `O(n)` ⇒ `O(n²)`. Có thể bỏ `all` để còn `O(n)`, nhưng giữ nguyên để làm "trọng tài" sát mô tả, ít suy luận.

**Nút thắt:** không cần mảng `watered` — hai người không bao giờ tưới trùng ngoại trừ cây giữa, xác định được trước.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight:** vì cùng tốc độ, Alice tưới đúng các cây `0..⌈n/2⌉−1` (trừ có thể cây giữa), Bob tưới `n−1..⌊n/2⌋`; hai quá trình **độc lập** — chỉ có cây giữa (nếu `n` lẻ) cần so sánh nước hiện có.

**Hai con trỏ:** `i = 0`, `j = n − 1`; trong khi `i < j`: Alice xử lý `i`, Bob xử lý `j`, `i += 1`, `j -= 1`. Sau vòng, nếu `i == j`: người có nước `max(wa, wb)` (hòa ⇒ Alice — nhưng hòa thì ai tưới cũng có cùng lượng nước nên kết quả **số lần nạp** như nhau) tưới; nạp nếu `max(wa, wb) < plants[i]`.

**Quan sát đáng chú ý:** với câu hỏi "số lần nạp", quy tắc phá hòa **không ảnh hưởng** kết quả — chỉ cần so `max(wa, wb)` với nhu cầu của cây giữa.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Hai con trỏ (code nộp)

In [6]:
def refill_two_pointers(plants, capacityA, capacityB):
    i, j = 0, len(plants) - 1
    wa, wb = capacityA, capacityB
    ans = 0
    while i < j:
        if wa < plants[i]:
            ans += 1; wa = capacityA
        wa -= plants[i]
        if wb < plants[j]:
            ans += 1; wb = capacityB
        wb -= plants[j]
        i += 1; j -= 1
    if i == j and max(wa, wb) < plants[i]:
        ans += 1
    return ans

for args, want in examples:
    print(refill_two_pointers(*args), "kỳ vọng", want)

1 kỳ vọng 1
2 kỳ vọng 2
0 kỳ vọng 0


Code nộp LeetCode của Cách 2: xem mục 9.

**Độ phức tạp:** `O(n)` thời gian, `O(1)` bộ nhớ.

### 7.2 Cách 3 — Tách hai nửa + cây giữa

Dùng `water_sequence` (mục 3.1) cho nửa trái (Alice) và nửa phải **đảo ngược** (Bob); xử lý cây giữa bằng lượng nước còn lại.

In [7]:
def refill_halves(plants, capacityA, capacityB):
    n = len(plants); m = n // 2
    ra, wa = water_sequence(plants[:m], capacityA)
    rb, wb = water_sequence(reversed(plants[n - m:]), capacityB)
    mid = 0
    if n % 2 == 1:
        mid = 1 if max(wa, wb) < plants[m] else 0
    return ra + rb + mid

for args, want in examples:
    print(refill_halves(*args), "kỳ vọng", want)

1 kỳ vọng 1
2 kỳ vọng 2
0 kỳ vọng 0


**Độ phức tạp:** `O(n)`, `O(n)` cho lát cắt (có thể dùng `islice` để `O(1)`). Ưu điểm: tái sử dụng hàm đã kiểm chứng, tách bạch "phần độc lập" và "phần tương tác".

### 7.3 Cách 4 — Mô phỏng ghi nhận ai tưới cây nào

Biến thể của Cách 2 trả về thêm danh sách "người tưới" cho mỗi cây và các thời điểm nạp — hữu ích để kiểm tra và cho các biến thể hỏi "ai tưới cây giữa", "tổng nước đã dùng"...

In [8]:
def refill_with_log(plants, capacityA, capacityB):
    n = len(plants); who = [None] * n; refill_at = []
    i, j, wa, wb = 0, n - 1, capacityA, capacityB
    while i < j:
        if wa < plants[i]: refill_at.append(("A", i)); wa = capacityA
        wa -= plants[i]; who[i] = "A"
        if wb < plants[j]: refill_at.append(("B", j)); wb = capacityB
        wb -= plants[j]; who[j] = "B"
        i += 1; j -= 1
    if i == j:
        if wa >= wb:
            if wa < plants[i]: refill_at.append(("A", i)); wa = capacityA
            wa -= plants[i]; who[i] = "A"
        else:
            if wb < plants[i]: refill_at.append(("B", i)); wb = capacityB
            wb -= plants[i]; who[i] = "B"
    return len(refill_at), who, refill_at

for args, want in examples + [(([1, 2, 4, 2, 1], 4, 5), 1)]:
    cnt, who, ra = refill_with_log(*args)
    print(f"{args}: {cnt} lần nạp (kỳ vọng {want}), người tưới {who}, nạp tại {ra}")

([2, 2, 3, 3], 5, 5): 1 lần nạp (kỳ vọng 1), người tưới ['A', 'A', 'B', 'B'], nạp tại [('B', 2)]
([2, 2, 3, 3], 3, 4): 2 lần nạp (kỳ vọng 2), người tưới ['A', 'A', 'B', 'B'], nạp tại [('A', 1), ('B', 2)]
([5], 10, 8): 0 lần nạp (kỳ vọng 0), người tưới ['A'], nạp tại []
([1, 2, 4, 2, 1], 4, 5): 1 lần nạp (kỳ vọng 1), người tưới ['A', 'A', 'B', 'B', 'B'], nạp tại [('B', 2)]


## 8. TRACE từng bước

Trace Cách 2: in `i`, `j`, nước của mỗi người trước/sau và số lần nạp.

In [9]:
def trace_two_pointers(plants, A, B):
    i, j, wa, wb, ans = 0, len(plants) - 1, A, B, 0
    print(f"plants = {plants}, A = {A}, B = {B}")
    while i < j:
        sa = "nạp" if wa < plants[i] else "   "
        if wa < plants[i]: ans += 1; wa = A
        wa -= plants[i]
        sb = "nạp" if wb < plants[j] else "   "
        if wb < plants[j]: ans += 1; wb = B
        wb -= plants[j]
        print(f"  Alice cây {i} (cần {plants[i]}) {sa} -> còn {wa:>2} | Bob cây {j} (cần {plants[j]}) {sb} -> còn {wb:>2} | tổng nạp {ans}")
        i += 1; j -= 1
    if i == j:
        who = "Alice" if wa >= wb else "Bob"; w = max(wa, wb)
        extra = 1 if w < plants[i] else 0; ans += extra
        print(f"  cây giữa {i} (cần {plants[i]}): Alice {wa}, Bob {wb} -> {who} tưới, {'nạp' if extra else 'không nạp'} | tổng nạp {ans}")
    return ans

for args, _ in examples:
    trace_two_pointers(*args)
trace_two_pointers([3, 1, 4, 1, 5, 9, 2], 6, 9)

plants = [2, 2, 3, 3], A = 5, B = 5
  Alice cây 0 (cần 2)     -> còn  3 | Bob cây 3 (cần 3)     -> còn  2 | tổng nạp 0
  Alice cây 1 (cần 2)     -> còn  1 | Bob cây 2 (cần 3) nạp -> còn  2 | tổng nạp 1
plants = [2, 2, 3, 3], A = 3, B = 4
  Alice cây 0 (cần 2)     -> còn  1 | Bob cây 3 (cần 3)     -> còn  1 | tổng nạp 0
  Alice cây 1 (cần 2) nạp -> còn  1 | Bob cây 2 (cần 3) nạp -> còn  1 | tổng nạp 2
plants = [5], A = 10, B = 8
  cây giữa 0 (cần 5): Alice 10, Bob 8 -> Alice tưới, không nạp | tổng nạp 0
plants = [3, 1, 4, 1, 5, 9, 2], A = 6, B = 9
  Alice cây 0 (cần 3)     -> còn  3 | Bob cây 6 (cần 2)     -> còn  7 | tổng nạp 0
  Alice cây 1 (cần 1)     -> còn  2 | Bob cây 5 (cần 9) nạp -> còn  0 | tổng nạp 1
  Alice cây 2 (cần 4) nạp -> còn  2 | Bob cây 4 (cần 5) nạp -> còn  4 | tổng nạp 3
  cây giữa 3 (cần 1): Alice 2, Bob 4 -> Bob tưới, không nạp | tổng nạp 3


3

## 9. Code nộp LeetCode

In [10]:
class Solution:
    def minimumRefill(self, plants: List[int], capacityA: int, capacityB: int) -> int:
        i, j = 0, len(plants) - 1
        a, b = capacityA, capacityB
        ans = 0
        while i < j:
            if a < plants[i]:
                ans += 1
                a = capacityA
            a -= plants[i]
            if b < plants[j]:
                ans += 1
                b = capacityB
            b -= plants[j]
            i += 1
            j -= 1
        if i == j and max(a, b) < plants[i]:   # cây giữa: người nhiều nước hơn tưới
            ans += 1
        return ans

sol = Solution()
for args, want in examples:
    got = sol.minimumRefill(*args)
    print(got, "OK" if got == want else "SAI")

1 OK
2 OK
0 OK


## 10. Vì sao thuật toán đúng

**Bổ đề 1 (lịch tưới).** Ở bước thời gian `t` (`t = 0, 1, …`), Alice tưới cây `t` và Bob tưới cây `n − 1 − t`, cho tới khi hai chỉ số gặp/vượt nhau. Lý do: mỗi cây tốn đúng một đơn vị thời gian và nạp là tức thời, nên tốc độ hai người bằng nhau và không phụ thuộc lượng nước.

**Bổ đề 2 (độc lập).** Với `t < n − 1 − t`, hai người tưới hai cây khác nhau; lượng nước của mỗi người chỉ phụ thuộc chuỗi cây **của chính họ** ⇒ đếm nạp riêng rẽ đúng như bài 2079 (không có quãng đường, chỉ đếm số lần nạp).

**Bổ đề 3 (cây giữa).** Nếu `n` lẻ, ở bước `t = (n−1)/2` cả hai tới cây giữa. Theo luật, người có nước `max(wa, wb)` tưới (hòa ⇒ Alice, cùng lượng nước). Người đó nạp ⇔ `max(wa, wb) < plants[mid]`. Người còn lại không làm gì. ⇒ cộng `[max(wa, wb) < plants[mid]]`. ✓

Vòng `while i < j` duyệt đúng các cặp của Bổ đề 2; điều kiện `i == j` sau vòng ⇔ `n` lẻ. ✓

## 11. Phân tích Time / Space

`n ≤ 10⁵`.

| Cách | Time | Space |
|---|---|---|
| 1. Mô phỏng theo thời gian | `O(n²)` (do `all`) | `O(n)` |
| 2. Hai con trỏ | `O(n)` | `O(1)` |
| 3. Hai nửa + cây giữa | `O(n)` | `O(n)` lát cắt |
| 4. Có ghi log | `O(n)` | `O(n)` |

In [11]:
rng = random.Random(1)
big = [rng.randint(1, 10**6) for _ in range(10**5 + 1)]
for A, B in [(10**6, 10**6), (3 * 10**6, 10**6), (10**9, 10**9)]:
    res = []
    for name, fn in [("two_pointers", refill_two_pointers), ("halves", refill_halves), ("submit", Solution().minimumRefill)]:
        t0 = time.perf_counter(); r = fn(big, A, B); res.append(r)
        print(f"A = {A:>10}, B = {B:>10} {name:12s}: {r:>6}  {(time.perf_counter() - t0) * 1000:.1f} ms")
    assert len(set(res)) == 1
t0 = time.perf_counter(); refill_time_sim(big[:3001], 10**6, 10**6); print(f"mô phỏng thời gian n = 3001: {time.perf_counter() - t0:.2f}s")

A =    1000000, B =    1000000 two_pointers:  66676  6.8 ms
A =    1000000, B =    1000000 halves      :  66676  4.5 ms
A =    1000000, B =    1000000 submit      :  66676  7.6 ms
A =    3000000, B =    1000000 two_pointers:  42742  8.6 ms
A =    3000000, B =    1000000 halves      :  42742  5.8 ms
A =    3000000, B =    1000000 submit      :  42742  8.9 ms
A = 1000000000, B = 1000000000 two_pointers:     49  6.9 ms
A = 1000000000, B = 1000000000 halves      :     49  3.9 ms
A = 1000000000, B = 1000000000 submit      :     49  6.9 ms
mô phỏng thời gian n = 3001: 0.00s


## 12. Bộ test

In [12]:
fns = [refill_two_pointers, refill_halves, lambda p, a, b: refill_with_log(p, a, b)[0], Solution().minimumRefill]

for args, want in examples:
    assert refill_time_sim(*args) == want
    for fn in fns:
        assert fn(*args) == want

edge = [
    (([1], 1, 1), 0), (([3], 3, 5), 0),
    (([2, 2], 2, 2), 0),                     # mỗi người một cây
    (([4, 1, 4], 4, 4), 1),                  # giữa: A còn 0, B còn 0 -> nạp
    (([4, 1, 4], 5, 4), 0),                  # giữa: A còn 1 >= 1
    (([1, 5, 1], 5, 5), 1),                  # cả hai còn 4 < 5 -> một lần nạp (không phải hai)
    (([1, 2, 4, 2, 1], 4, 5), 1),
    (([5, 5, 5, 5, 5, 5], 5, 5), 4),
]
for args, want in edge:
    assert refill_time_sim(*args) == want, args
    for fn in fns:
        assert fn(*args) == want, (fn, args)

rng = random.Random(2024)
for _ in range(3000):
    n = rng.randint(1, 20)
    plants = [rng.randint(1, 9) for _ in range(n)]
    A = rng.randint(max(plants), max(plants) + 6); B = rng.randint(max(plants), max(plants) + 6)
    want = refill_time_sim(plants, A, B)
    for fn in fns:
        assert fn(plants, A, B) == want, (fn, plants, A, B)
print("ví dụ + edge + 3000 bộ ngẫu nhiên khớp mô phỏng theo thời gian")

ví dụ + edge + 3000 bộ ngẫu nhiên khớp mô phỏng theo thời gian


## 13. Lỗi người mới hay gặp

### Lỗi 1 — So sánh **dung tích** thay vì nước hiện có ở cây giữa

In [13]:
def wrong_compare_capacity(plants, A, B):
    i, j, a, b, ans = 0, len(plants) - 1, A, B, 0
    while i < j:
        if a < plants[i]: ans += 1; a = A
        a -= plants[i]
        if b < plants[j]: ans += 1; b = B
        b -= plants[j]
        i += 1; j -= 1
    if i == j:
        w = a if A >= B else b                                   # SAI: chọn người theo dung tích
        if w < plants[i]: ans += 1
    return ans

for args in [([1, 2, 4, 2, 1], 6, 5), ([5, 1, 5], 9, 6)]:
    print(f"{args}: sai = {wrong_compare_capacity(*args)} | đúng = {refill_time_sim(*args)}")

([1, 2, 4, 2, 1], 6, 5): sai = 1 | đúng = 1
([5, 1, 5], 9, 6): sai = 0 | đúng = 0


Với hai bộ thử trên, người có dung tích lớn hơn **tình cờ** cũng là người còn nhiều nước hơn lúc gặp nhau, nên lỗi không lộ — đây là lý do test tay dễ bỏ sót. Tìm phản ví dụ thật bằng tìm kiếm ngẫu nhiên:

In [14]:
rng = random.Random(1)
for _ in range(20000):
    plants = [rng.randint(1, 6) for _ in range(rng.choice([3, 5, 7]))]
    A = rng.randint(max(plants), 10); B = rng.randint(max(plants), 10)
    if wrong_compare_capacity(plants, A, B) != refill_time_sim(plants, A, B):
        print(f"phản ví dụ: plants = {plants}, A = {A}, B = {B} | sai = {wrong_compare_capacity(plants, A, B)} | đúng = {refill_time_sim(plants, A, B)}")
        trace_two_pointers(plants, A, B)
        break

phản ví dụ: plants = [2, 2, 5, 2, 1, 2, 5], A = 9, B = 6 | sai = 2 | đúng = 1
plants = [2, 2, 5, 2, 1, 2, 5], A = 9, B = 6
  Alice cây 0 (cần 2)     -> còn  7 | Bob cây 6 (cần 5)     -> còn  1 | tổng nạp 0
  Alice cây 1 (cần 2)     -> còn  5 | Bob cây 5 (cần 2) nạp -> còn  4 | tổng nạp 1
  Alice cây 2 (cần 5)     -> còn  0 | Bob cây 4 (cần 1)     -> còn  3 | tổng nạp 1
  cây giữa 3 (cần 2): Alice 0, Bob 3 -> Bob tưới, không nạp | tổng nạp 1


Đề nói "người có **nhiều nước hơn trong bình hiện tại**". **Sửa:** `max(wa, wb)`.

### Lỗi 2 — Xử lý cây giữa trong vòng lặp `while i <= j` (tưới hai lần)

In [15]:
def wrong_le_loop(plants, A, B):
    i, j, a, b, ans = 0, len(plants) - 1, A, B, 0
    while i <= j:                                               # SAI: khi i == j cả hai cùng tưới
        if a < plants[i]: ans += 1; a = A
        a -= plants[i]
        if b < plants[j]: ans += 1; b = B
        b -= plants[j]
        i += 1; j -= 1
    return ans

for args, want in examples + [(([1, 5, 1], 5, 5), 1), (([4, 1, 4], 4, 4), 1)]:
    print(f"{args}: sai = {wrong_le_loop(*args)} | đúng = {want}")

([2, 2, 3, 3], 5, 5): sai = 1 | đúng = 1
([2, 2, 3, 3], 3, 4): sai = 2 | đúng = 2
([5], 10, 8): sai = 0 | đúng = 0
([1, 5, 1], 5, 5): sai = 2 | đúng = 1
([4, 1, 4], 4, 4): sai = 2 | đúng = 1


Cây giữa chỉ được tưới **một** lần, bởi một người. `[1, 5, 1]`, `A = B = 5`: cả hai còn `4 < 5`; chỉ người tưới nạp ⇒ `1`, không phải `2`. **Sửa:** `while i < j` rồi xử lý `i == j` riêng.

### Lỗi 3 — Nạp khi **vừa đủ** (`≤`)

In [16]:
def wrong_le_refill(plants, A, B):
    i, j, a, b, ans = 0, len(plants) - 1, A, B, 0
    while i < j:
        if a <= plants[i] and a != A: ans += 1; a = A            # SAI: vừa đủ vẫn tưới được
        a -= plants[i]
        if b <= plants[j] and b != B: ans += 1; b = B
        b -= plants[j]
        i += 1; j -= 1
    if i == j and max(a, b) <= plants[i]: ans += 1
    return ans

for args, want in examples + [(([2, 3, 3, 2], 5, 5), 0)]:
    print(f"{args}: sai = {wrong_le_refill(*args)} | đúng = {want}")

([2, 2, 3, 3], 5, 5): sai = 1 | đúng = 1
([2, 2, 3, 3], 3, 4): sai = 2 | đúng = 2
([5], 10, 8): sai = 0 | đúng = 0
([2, 3, 3, 2], 5, 5): sai = 2 | đúng = 0


"Must water the plant if they have enough" ⇒ đủ (kể cả vừa đúng) thì tưới. **Sửa:** `<`.

### Lỗi 4 — Cây giữa: cộng nạp nếu **bất kỳ** ai thiếu

In [17]:
def wrong_mid_any(plants, A, B):
    i, j, a, b, ans = 0, len(plants) - 1, A, B, 0
    while i < j:
        if a < plants[i]: ans += 1; a = A
        a -= plants[i]
        if b < plants[j]: ans += 1; b = B
        b -= plants[j]
        i += 1; j -= 1
    if i == j and min(a, b) < plants[i]: ans += 1               # SAI: người nhiều nước hơn mới tưới
    return ans

for args in [([1, 2, 4, 2, 1], 4, 5), ([1, 3, 1], 3, 4)]:
    print(f"{args}: sai = {wrong_mid_any(*args)} | đúng = {refill_time_sim(*args)}")

([1, 2, 4, 2, 1], 4, 5): sai = 1 | đúng = 1
([1, 3, 1], 3, 4): sai = 1 | đúng = 0


`[1, 3, 1]`, `A = 3`, `B = 4`: Alice còn `2`, Bob còn `3` — Bob tưới cây giữa (cần 3) mà không nạp; dùng `min` thì đếm nhầm một lần nạp. **Sửa:** `max(a, b)`.

### Lỗi 5 — Nạp bằng `+= capacity`

Giống bài 2079: bình chỉ chứa tối đa dung tích ⇒ nạp là **gán** `a = capacityA`. Viết `a += capacityA` làm nước "tích lũy" và đếm thiếu số lần nạp.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Hai tác nhân di chuyển **cùng tốc độ** từ hai đầu ⇒ hai con trỏ, chỉ va chạm ở giữa.
- Tách **phần độc lập** (mỗi người một nửa) khỏi **phần tương tác** (cây giữa) — viết hàm cho phần độc lập.
- Đọc kỹ tiêu chí phá hòa: "nước hiện có" ≠ "dung tích".
- Họ bài: 2079 *Watering Plants*, 1423 *Maximum Points You Can Obtain from Cards*, 881 *Boats to Save People*, 2337 *Move Pieces to Obtain a String*.

## 15. Biến thể của bài

1. **Tốc độ khác nhau** (Alice tưới 2 cây mỗi đơn vị thời gian, Bob 1): điểm gặp không còn ở giữa — mô phỏng theo thời gian với con trỏ (Bài 5).
2. **Hỏi ai tưới mỗi cây / tổng nước mỗi người dùng**: Cách 4 có log.
3. **Có quãng đường tới nguồn nước** (như 2079): mỗi người có sông riêng ở hai đầu; cộng `2·khoảng cách` mỗi lần nạp.
4. **Ba người** (thêm Carol bắt đầu ở giữa đi hai phía): chia mảng thành các đoạn độc lập theo thời gian.

In [18]:
def steps_with_rivers(plants, A, B):
    """Biến thể 3 (tự đặt): Alice có sông ở x = -1, Bob ở x = n; đếm tổng bước đi của cả hai.
    Alice đi từ -1 tới cây cuối của mình, mỗi lần nạp trước cây i tốn thêm 2i (như 2079); Bob đối xứng."""
    n = len(plants); m = n // 2
    def side(seq, cap):
        steps, water = 0, cap
        for k, need in enumerate(seq):
            if water < need: steps += 2 * k; water = cap
            water -= need
        return steps + len(seq), water
    sa, wa = side(plants[:m], A); sb, wb = side(plants[n - m:][::-1], B)
    if n % 2:
        if wa >= wb:
            sa += 1 + (2 * m if wa < plants[m] else 0)
        else:
            sb += 1 + (2 * m if wb < plants[m] else 0)
    return sa + sb

print("biến thể sông hai đầu:", [steps_with_rivers(*args) for args, _ in examples])

biến thể sông hai đầu: [6, 8, 1]


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mô phỏng thời gian | nghĩa đen | `O(n²)` | `O(n)` | kiểm chứng; tốc độ khác nhau |
| 2. Hai con trỏ | độc lập + giữa | `O(n)` | `O(1)` | **nộp bài** |
| 3. Hai nửa | tái sử dụng hàm | `O(n)` | `O(n)` | code mô-đun |
| 4. Có log | ghi người tưới | `O(n)` | `O(n)` | cần chi tiết |

## 17. Bài tập (từ dễ đến khó)

**Bài 1 (đoán output):** `[1, 1, 1, 1]`, `1`, `1`? `[3, 2, 3]`, `3`, `3`? `[2, 4, 5, 1, 2]`, `5`, `7`?

**Bài 2 (tính tay):** `plants = [3, 1, 4, 1, 5, 9, 2]`, `A = 6`, `B = 9`. Viết bảng từng bước và cây giữa thuộc về ai.

**Bài 3 (điền TODO):** hoàn thiện `who_waters_middle(plants, A, B)` trả về `"A"`, `"B"` hoặc `None` (khi `n` chẵn).

**Bài 4 (tự viết lại):** viết `total_water_used(plants, A, B)` trả về `(nước Alice đã dùng, nước Bob đã dùng)`; kiểm tra tổng bằng `sum(plants)`.

**Bài 5 (mini challenge):** Alice tưới với tốc độ `sa` cây/đơn vị thời gian, Bob `sb` cây/đơn vị thời gian (số nguyên dương). Trong một đơn vị thời gian, mỗi người tưới lần lượt tối đa `sa`/`sb` cây kế tiếp **chưa tưới**; Alice hành động trước trong mỗi đơn vị thời gian. Đếm số lần nạp. Viết mô phỏng `O(n)` bằng hai con trỏ. (Biến thể tự đặt — luật va chạm tự đặt; đối chiếu lại đề gốc nếu dùng làm đề.)

In [19]:
def who_waters_middle(plants, A, B):
    # TODO: mô phỏng hai nửa, so nước còn lại; hòa -> "A"
    pass

## 18. Gợi ý 3 mức

**Bài 1:** (1) `[1,1,1,1]`, `1`, `1`: mỗi người tưới hai cây, cây thứ hai cần nạp ⇒ `2`. (2) `[3,2,3]`, `3`, `3`: sau cây đầu cả hai còn `0`; cây giữa cần `2` ⇒ nạp `1`. (3) `[2,4,5,1,2]`, `5`, `7`: Alice `2` rồi `4` (nạp) còn `1`; Bob `2`, `1` còn `4`; giữa cần `5` > `4` ⇒ nạp ⇒ `2`.

**Bài 2:** (1) Dùng `trace_two_pointers`. (2) Cặp `(0,6)`, `(1,5)`, `(2,4)`, giữa `3`. (3) Xem output ở mục 8.

**Bài 3:** (1) `n` chẵn ⇒ `None`. (2) `water_sequence` cho hai nửa lấy nước còn. (3) `"A" if wa >= wb else "B"`.

**Bài 4:** (1) Dùng Cách 4 (log người tưới). (2) Cộng `plants[i]` cho người tưới cây `i`. (3) Tổng hai người `= sum(plants)`.

**Bài 5:** (1) Hai con trỏ `i`, `j`. (2) Mỗi đơn vị thời gian: Alice tưới tối đa `sa` cây với `i ≤ j`, rồi Bob tối đa `sb` cây với `j ≥ i`. (3) Dừng khi `i > j`.

## 19. Đáp án tham khảo

In [20]:
# Bài 1
assert [Solution().minimumRefill(*a) for a in [([1, 1, 1, 1], 1, 1), ([3, 2, 3], 3, 3), ([2, 4, 5, 1, 2], 5, 7)]] == [2, 1, 2]
# Bài 2
r2 = trace_two_pointers([3, 1, 4, 1, 5, 9, 2], 6, 9)
assert r2 == refill_time_sim([3, 1, 4, 1, 5, 9, 2], 6, 9)

# Bài 3
def who_waters_middle_ans(plants, A, B):
    n = len(plants)
    if n % 2 == 0: return None
    m = n // 2
    _, wa = water_sequence(plants[:m], A); _, wb = water_sequence(reversed(plants[m + 1:]), B)
    return "A" if wa >= wb else "B"

# Bài 4
def total_water_used_ans(plants, A, B):
    _, who, _ = refill_with_log(plants, A, B)
    ua = sum(p for p, w in zip(plants, who) if w == "A"); ub = sum(p for p, w in zip(plants, who) if w == "B")
    return ua, ub

rng = random.Random(41)
for _ in range(500):
    plants = [rng.randint(1, 9) for _ in range(rng.randint(1, 15))]
    A = rng.randint(max(plants), 14); B = rng.randint(max(plants), 14)
    ua, ub = total_water_used_ans(plants, A, B); assert ua + ub == sum(plants)
    w = who_waters_middle_ans(plants, A, B)
    if w is not None: assert refill_with_log(plants, A, B)[1][len(plants) // 2] == w
print("Bài 3:", who_waters_middle_ans([1, 2, 4, 2, 1], 4, 5), "| Bài 4:", total_water_used_ans([2, 2, 3, 3], 3, 4))
print("Bài 1-4: OK")

plants = [3, 1, 4, 1, 5, 9, 2], A = 6, B = 9
  Alice cây 0 (cần 3)     -> còn  3 | Bob cây 6 (cần 2)     -> còn  7 | tổng nạp 0
  Alice cây 1 (cần 1)     -> còn  2 | Bob cây 5 (cần 9) nạp -> còn  0 | tổng nạp 1
  Alice cây 2 (cần 4) nạp -> còn  2 | Bob cây 4 (cần 5) nạp -> còn  4 | tổng nạp 3
  cây giữa 3 (cần 1): Alice 2, Bob 4 -> Bob tưới, không nạp | tổng nạp 3
Bài 3: B | Bài 4: (4, 6)
Bài 1-4: OK


In [21]:
# Bài 5 — tốc độ khác nhau
def refill_speeds(plants, A, B, sa, sb):
    i, j, a, b, ans = 0, len(plants) - 1, A, B, 0
    while i <= j:
        for _ in range(sa):
            if i > j: break
            if a < plants[i]: ans += 1; a = A
            a -= plants[i]; i += 1
        for _ in range(sb):
            if j < i: break
            if b < plants[j]: ans += 1; b = B
            b -= plants[j]; j -= 1
    return ans

def refill_speeds_brute(plants, A, B, sa, sb):
    n = len(plants); done = [False] * n; a, b, ans = A, B, 0
    while not all(done):
        for _ in range(sa):
            k = next((t for t in range(n) if not done[t]), None)
            if k is None: break
            if a < plants[k]: ans += 1; a = A
            a -= plants[k]; done[k] = True
        for _ in range(sb):
            k = next((t for t in range(n - 1, -1, -1) if not done[t]), None)
            if k is None: break
            if b < plants[k]: ans += 1; b = B
            b -= plants[k]; done[k] = True
    return ans

rng = random.Random(55)
for _ in range(1000):
    plants = [rng.randint(1, 9) for _ in range(rng.randint(1, 15))]
    A = rng.randint(max(plants), 14); B = rng.randint(max(plants), 14); sa, sb = rng.randint(1, 3), rng.randint(1, 3)
    assert refill_speeds(plants, A, B, sa, sb) == refill_speeds_brute(plants, A, B, sa, sb)
print("Bài 5: khớp mô phỏng; ví dụ [2,2,3,3], A = 3, B = 4, sa = 2, sb = 1 ->", refill_speeds([2, 2, 3, 3], 3, 4, 2, 1))

Bài 5: khớp mô phỏng; ví dụ [2,2,3,3], A = 3, B = 4, sa = 2, sb = 1 -> 2


Lưu ý Bài 5: luật va chạm ở đây (Alice hành động trước trong mỗi đơn vị thời gian) là **tự đặt**, khác luật "ai nhiều nước hơn" của đề gốc — với `sa = sb = 1` và `n` lẻ, luật này luôn giao cây giữa cho Alice nên kết quả có thể khác bài gốc.

## 20. Template nhớ nhanh + Checklist

```python
# Hai người tưới từ hai đầu, cùng tốc độ
i, j, a, b, ans = 0, n - 1, A, B, 0
while i < j:
    if a < plants[i]: ans += 1; a = A      # nạp khi THIẾU (strict), gán đầy
    a -= plants[i]
    if b < plants[j]: ans += 1; b = B
    b -= plants[j]
    i += 1; j -= 1
if i == j and max(a, b) < plants[i]:       # cây giữa: người nhiều nước hơn tưới
    ans += 1
```

**Checklist trước khi submit:**
- [ ] `while i < j` (không `<=`), cây giữa xử lý riêng.
- [ ] Cây giữa so `max(a, b)` — nước **hiện có**, không phải dung tích.
- [ ] Nạp khi `water < need` (vừa đủ thì không nạp).
- [ ] Nạp là gán `= capacity`.
- [ ] Test `n = 1`, `n` chẵn, `n` lẻ với hòa nước.